# Lesson 02 - Istraživanje Microsoft Agent Frameworka

**Microsoft Agent Framework (MAF)** je jedinstveni okvir za izgradnju AI agenata. Pruža čistu, sastavljivu arhitekturu s četiri osnovne sastavnice:

- **Klijent** – povezuje se na krajnju točku AI modela i upravlja komunikacijom
- **Agent** – obuhvaća klijenta s uputama i definicijama alata
- **Alati** – proširuju mogućnosti agenta prilagođenim funkcijama koje model može pozvati
- **Sesija** – održava povijest razgovora za višekratne interakcije

U ovoj lekciji izgradit ćemo **agenta za rezervaciju putovanja** koji provjerava dostupnost destinacija koristeći ove koncepte.


## Postavljanje


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Razumijevanje arhitekture Agent Frameworka

Microsoft Agent Framework slijedi slojevitu arhitekturu:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klijent** – `AzureAIProjectAgentProvider` se povezuje s Azure OpenAI implementacijom. Rukuje autentifikacijom, oblikovanjem zahtjeva i parsiranjem odgovora.
2. **Agent** – Kreiran iz klijenta putem `provider.create_agent()`, agent kombinira pristup modelu s uputama (sistemski prompt) i alatima.
3. **Alati** – Python funkcije ukrašene s `@tool` koje agent može pozvati da izvrši radnje ili dohvati podatke.
4. **Sesija** – Objekt `AgentSession` (kreiran putem `agent.create_session()`) koji pohranjuje povijest razgovora, omogućujući višekratni dijalog u kojem agent pamti prethodni kontekst.

Izgradimo svaki sloj korak po korak.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dodavanje alata s @tool dekoratorom

Alati omogućuju agentima da poduzimaju radnje izvan generiranja teksta. `@tool` dekorator pretvara uobičajenu Python funkciju u nešto što agent može pozvati.

Ključne točke:
- Koristite `Annotated[type, "description"]` kako bi model razumio svaki parametar.
- Docstring postaje opis alata koji model vidi.
- `approval_mode="never_require"` znači da se alat pokreće automatski bez potvrde korisnika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Izrada agenta s alatima

Sada kombiniramo klijenta, upute i alate u agenta. `instructions` djeluju kao sustavni upit — oni definiraju osobnost i ponašanje agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Višekratni razgovori sa sesijama

`AgentSession` (kreiran putem `agent.create_session()`) prati sve poruke u razgovoru. Prosljeđivanjem iste sesije svakom pozivu `agent.run()`, agent ima pristup cjelokupnoj povijesti razgovora i može se pozivati na ranije poruke.

Prosljeđujemo `tools=[check_destination_availability]` kako bi agent mogao pozivati naš alat za provjeru dostupnosti tijekom svakog koraka.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sažetak

U ovoj lekciji istražili ste četiri stupa Microsoft Agent Frameworka:

| Pojam | Što ste naučili |
|---------|------------------|
| **Klijent** | `AzureAIProjectAgentProvider` se povezuje na Azure OpenAI s autentifikacijom temeljenom na vjerodajnicama |
| **Agent** | `provider.create_agent()` povezuje model s uputama i imenom |
| **Alati** | Dekorator `@tool` izlaže Python funkcije koje agent može pozivati |
| **Sesija** | `agent.create_session()` održava povijest razgovora kroz više okretaja |

Ovi građevni blokovi tvore agente koji mogu voditi prirodne razgovore, pozivati vanjske funkcije i održavati kontekst — temelj za naprednije agentne obrasce u kasnijim lekcijama.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj dokument je preveden korištenjem AI usluge prevođenja [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, molimo imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakva nesporazuma ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
